# Parking Sign Detection - Experiment 7: BEST CONFIGURATION

Combines all winning elements from Experiments C and D for optimal quality.

**Configuration:**
- Model: YOLO11m (proven best balance of speed/accuracy)
- Negatives: Controlled (~20%) for stability
- Augmentation: Full suite (mosaic, mixup, copy-paste, etc.)
- Training: Extended epochs, increased patience for maximum quality

**Expected Results:** mAP50-95 ~0.99 (based on Experiments C & D)

In [ ]:
# === Cell 1: Install dependencies ===
# Install ultralytics first (with its numpy 2.x deps)
!pip install ultralytics -q

# Then downgrade numpy for torchvision compatibility
!pip install "numpy<2" --force-reinstall -q

from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import yaml
import os
from pathlib import Path

In [ ]:
# === Cell 2: Setup paths ===

# Dataset path (Kaggle input)
DATASET_PATH = Path('/kaggle/input/parking-sign-detection-coco-dataset/parking-sign-detection-coco-dataset')

# Working directory
WORKING_PATH = Path('/kaggle/working')
OUTPUT_PATH = WORKING_PATH

print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

# List dataset contents
if DATASET_PATH.exists():
    print(f"\nContents:")
    for item in sorted(DATASET_PATH.iterdir()):
        print(f"  {item.name}")

In [ ]:
# === Cell 3: Load and inspect original data.yaml ===

ORIGINAL_DATA_YAML = DATASET_PATH / 'data.yaml'

with open(ORIGINAL_DATA_YAML, 'r') as f:
    original_config = yaml.safe_load(f)

print("Original config:")
print(original_config)

# Count images in each split
for split in ['train', 'valid', 'test']:
    split_path = DATASET_PATH / split / 'images'
    if split_path.exists():
        n_images = len(list(split_path.glob('*.jpg')))
        print(f"\n{split} ({split}): {n_images} images")

In [ ]:
# === Cell 4: Create filtered dataset with controlled negatives (~20%) ===

FILTERED_PATH = WORKING_PATH / 'dataset_controlled_negatives'
FILTERED_DATA_YAML = FILTERED_PATH / 'data.yaml'

# Target negative ratio
TARGET_NEGATIVE_RATIO = 0.20  # 20% negatives

def create_filtered_dataset(original_path, filtered_path, target_neg_ratio=0.20):
    """Create dataset with controlled negative samples."""
    
    if filtered_path.exists():
        print(f"Filtered dataset already exists at {filtered_path}")
        return
    
    print(f"Creating filtered dataset at {filtered_path}")
    print(f"Target negative ratio: {target_neg_ratio:.1%}")
    
    filtered_path.mkdir(parents=True, exist_ok=True)
    
    for split in ['train', 'valid', 'test']:
        # Create directories
        (filtered_path / split / 'images').mkdir(parents=True, exist_ok=True)
        (filtered_path / split / 'labels').mkdir(parents=True, exist_ok=True)
        
        # Get all label files
        label_dir = original_path / split / 'labels'
        img_dir = original_path / split / 'images'
        
        if not label_dir.exists():
            continue
            
        label_files = list(label_dir.glob('*.txt'))
        
        # Separate positives and negatives
        positives = []
        negatives = []
        
        for label_file in label_files:
            with open(label_file, 'r') as f:
                content = f.read().strip()
            
            # Empty file = negative (no objects)
            if not content or content.strip() == '':
                negatives.append(label_file)
            else:
                positives.append(label_file)
        
        # Calculate how many negatives to sample
        n_positives = len(positives)
        n_negatives_to_sample = int(n_positives * target_neg_ratio / (1 - target_neg_ratio))
        
        # For valid/test, keep all (no negatives sampling)
        if split != 'train':
            selected_labels = positives
        else:
            # Sample negatives for training
            n_negatives_to_sample = min(n_negatives_to_sample, len(negatives))
            sampled_negatives = list(pd.Series(negatives).sample(n_negatives_to_sample, random_state=42))
            selected_labels = positives + sampled_negatives
        
        # Copy files
        for label_file in selected_labels:
            # Copy label
            shutil.copy(label_file, filtered_path / split / 'labels' / label_file.name)
            
            # Copy corresponding image
            img_file = img_dir / (label_file.stem + '.jpg')
            if img_file.exists():
                shutil.copy(img_file, filtered_path / split / 'images' / img_file.name)
        
        print(f"{split}: {n_positives} positives + {len(negatives) if split == 'train' else 0} negatives sampled = {len(selected_labels)} total")
    
    # Create filtered data.yaml
    filtered_config = original_config.copy()
    filtered_config['path'] = str(filtered_path)
    filtered_config['train'] = str((filtered_path / 'train' / 'images'))
    filtered_config['val'] = str((filtered_path / 'valid' / 'images'))
    filtered_config['test'] = str((filtered_path / 'test' / 'images'))
    
    with open(filtered_path / 'data.yaml', 'w') as f:
        yaml.dump(filtered_config, f)
    
    print(f"\nFiltered data.yaml written to {filtered_path / 'data.yaml'}")

# Run the filtering
create_filtered_dataset(DATASET_PATH, FILTERED_PATH, TARGET_NEGATIVE_RATIO)

In [ ]:
# === Cell 5: Display final config ===

with open(FILTERED_DATA_YAML, 'r') as f:
    final_config = yaml.safe_load(f)

print("Final configuration:")
for key, value in final_config.items():
    print(f"  {key}: {value}")

In [ ]:
# === Cell 6: Training configuration ===

RUN_NAME = "parking_sign_best"

TRAIN_PARAMS = {
    # Data
    "data": str(FILTERED_DATA_YAML),
    
    # Extended training for maximum quality
    "epochs": 100,  # Increased from 80
    "patience": 12,  # Increased from 8 - allow more time for late improvements
    
    # Image settings
    "imgsz": 640,
    "batch": 16,  # Optimal for 2x Tesla T4
    "workers": 4,
    
    # Multi-GPU
    "device": "0,1",
    
    # Learning rate - cosine decay (proven effective)
    "cos_lr": True,
    "lr0": 0.005,
    "lrf": 0.005,
    
    # Optimizer
    "optimizer": "auto",  # AdamW works well
    "weight_decay": 0.0005,
    "momentum": 0.937,
    
    # Augmentation - FULL SUITE (from Experiment C)
    "mosaic": 1.0,
    "mixup": 0.08,
    "copy_paste": 0.05,
    "degrees": 8.0,      # Rotation
    "translate": 0.1,    # Translation
    "scale": 0.4,        # Scaling
    "shear": 2.0,        # Shear
    "perspective": 0.0001,
    "fliplr": 0.5,       # Horizontal flip
    "flipud": 0.0,       # No vertical flip (signs not usually upside down)
    # HSV color augmentation
    "hsv_h": 0.01,
    "hsv_s": 0.5,
    "hsv_v": 0.3,
    # Mosaic augmentation disabled for last N epochs
    "close_mosaic": 25,
    
    # Training settings
    "pretrained": True,
    "verbose": True,
    "save_period": 10,
    "exist_ok": True,
    
    # Output
    "project": str(OUTPUT_PATH / "runs"),
    "name": RUN_NAME,
}

print(f"Training config: {len(TRAIN_PARAMS)} parameters")
print(f"Model: YOLO11m | Max epochs: {TRAIN_PARAMS['epochs']}, Patience: {TRAIN_PARAMS['patience']}")
print(f"Image size: {TRAIN_PARAMS['imgsz']} | Batch: {TRAIN_PARAMS['batch']} | Device: {TRAIN_PARAMS['device']}")
print(f"Negative ratio: {TARGET_NEGATIVE_RATIO:.0%} | Augmentation: FULL")
print(f"\nTermination: early stop after {TRAIN_PARAMS['patience']} epochs without mAP50 improvement")

In [ ]:
# === Cell 7: Train model ===

print("="*60)
print("Training Experiment 7: BEST CONFIGURATION")
print("="*60)
print(f"\nConfiguration summary:")
print(f"  Model: YOLO11m (20M params - optimal balance)")
print(f"  Negatives: Controlled ({TARGET_NEGATIVE_RATIO:.0%} ratio)")
print(f"  Augmentation: Full suite (mosaic, mixup, copy-paste, etc.)")
print(f"  Training: {TRAIN_PARAMS['epochs']} max epochs, patience={TRAIN_PARAMS['patience']}")
print(f"\nExpected results: mAP50-95 ~0.99")
print("="*60)

# Load pretrained YOLO11m model
model = YOLO("yolo11m.pt")

# Train
train_results = model.train(**TRAIN_PARAMS)

In [ ]:
# === Cell 8: Load best model and evaluate ===

from ultralytics import YOLO

# Load best model
best_model_path = OUTPUT_PATH / "runs" / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(best_model_path)

print(f"Loaded best model from: {best_model_path}")

# Run validation on test set
print("\nEvaluating on test set...")
test_results = best_model.val(data=str(FILTERED_DATA_YAML), split="test")

# Extract metrics
test_mAP50 = test_results.box.map50
test_mAP50_95 = test_results.box.map
test_precision = test_results.box.mp
test_recall = test_results.box.mr

print(f"\n{'='*50}")
print(f"TEST SET RESULTS")
print(f"{'='*50}")
print(f"  Precision:  {test_precision:.4f}")
print(f"  Recall:     {test_recall:.4f}")
print(f"  mAP@50:     {test_mAP50:.4f}")
print(f"  mAP@50-95:  {test_mAP50_95:.4f}")
print(f"{'='*50}")

# Compare with previous experiments
print(f"\nComparison with previous experiments:")
print(f"  Exp C (YOLO11m, controlled neg):  mAP50-95=0.991")
print(f"  Exp D (YOLO11l, large model):    mAP50-95=0.990")
print(f"  This run:                        mAP50-95={test_mAP50_95:.3f}")

if test_mAP50_95 >= 0.99:
    print(f"\n✅ EXCELLENT - Target achieved! Model ready for deployment.")
elif test_mAP50_95 >= 0.98:
    print(f"\n✅ VERY GOOD - Near-production quality.")
elif test_mAP50_95 >= 0.95:
    print(f"\n⚠️  GOOD - May need targeted improvements for specific cases.")
else:
    print(f"\n❌ NEEDS IMPROVEMENT")

In [ ]:
# === Cell 9: Plot training curves ===

import pandas as pd
import matplotlib.pyplot as plt

# Read results.csv
results_csv = OUTPUT_PATH / "runs" / RUN_NAME / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    
    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{RUN_NAME} - Training Curves', fontsize=14, fontweight='bold')
    
    # Plot losses
    ax = axes[0, 0]
    ax.plot(df['epoch'], df['train/box_loss'], label='Box Loss', alpha=0.7)
    ax.plot(df['epoch'], df['train/cls_loss'], label='Cls Loss', alpha=0.7)
    ax.plot(df['epoch'], df['train/dfl_loss'], label='DFL Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot mAP
    ax = axes[0, 1]
    ax.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50', color='green')
    ax.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95', color='blue')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_title('Mean Average Precision')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Plot precision/recall
    ax = axes[1, 0]
    ax.plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='orange')
    ax.plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='red')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_title('Precision & Recall')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Plot validation vs training
    ax = axes[1, 1]
    ax.plot(df['epoch'], df['metrics/mAP50(B)'], label='Val mAP@50', color='green', alpha=0.7)
    ax.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='Val mAP@50-95', color='blue', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_title('All Metrics Summary')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(OUTPUT_PATH / 'training_curves.png', dpi=150)
    plt.show()
    
    print(f"\nTraining curves saved to: {OUTPUT_PATH / 'training_curves.png'}")
    
    # Print best metrics
    print(f"\nBest metrics achieved:")
    best_map50_idx = df['metrics/mAP50(B)'].idxmax()
    best_map50_95_idx = df['metrics/mAP50-95(B)'].idxmax()
    print(f"  Best mAP@50:    {df.loc[best_map50_idx, 'metrics/mAP50(B)']:.4f} (epoch {df.loc[best_map50_idx, 'epoch']})")
    print(f"  Best mAP@50-95: {df.loc[best_map50_95_idx, 'metrics/mAP50-95(B)']:.4f} (epoch {df.loc[best_map50_95_idx, 'epoch']})")

In [ ]:
# === Cell 10: Export model for deployment ===

print("Preparing model for deployment...")

# Model is already at: best.pt
# For different export formats:

# Export to ONNX (for CPU inference)
onnx_path = best_model.export(format='onnx', dynamic=True, simplify=True)
print(f"\nONNX model: {onnx_path}")

# Export to TensorRT (for NVIDIA GPU inference - faster)
# Note: Skip if TensorRT not available
try:
    trt_path = best_model.export(format='engine', half=True)
    print(f"TensorRT model: {trt_path}")
except Exception as e:
    print(f"TensorRT export skipped: {e}")

print(f"\n{'='*50}")
print(f"DEPLOYMENT SUMMARY")
print(f"{'='*50}")
print(f"\nBest model (PyTorch): {best_model_path}")
print(f"ONNX model:          {onnx_path}")
print(f"\nRecommended for:")
print(f"  - GPU serving:     Use {best_model_path}")
print(f"  - CPU/edge:        Use {onnx_path}")
print(f"  - Web apps:        Use {onnx_path} with ONNX Runtime")
print(f"\nTest set performance:")
print(f"  mAP@50-95:         {test_mAP50_95:.4f}")
print(f"  Precision:         {test_precision:.4f}")
print(f"  Recall:            {test_recall:.4f}")